# 3.4 Demo - Shuffle
**Lesson:** Shuffles

A **shuffle** redistributes rows across the cluster so that rows with the same key end up in the same partition. It is needed for `groupBy`, `join`, `distinct`, `orderBy`, `repartition`, and window functions. It costs disk I/O (shuffle files), network, and serialization, and it creates a **stage boundary**.

Tools to control it:
1. **Avoid** it: broadcast the small side of a join, pre-aggregate, filter early
2. **Size** it: `spark.sql.shuffle.partitions`, or let **AQE** coalesce partitions at runtime
3. **Reuse** it: tables clustered or bucketed on the join key (advanced)

In [ ]:
from adelab.classroom import setup
from adelab import perf
from pyspark.sql import functions as F

DA = setup("3.4", reset=True)

(spark.range(4_000_000)
      .withColumn("customer_id", (F.rand(seed=1) * 200_000).cast("int"))
      .withColumn("product_id", (F.rand(seed=2) * 1_000).cast("int"))
      .withColumn("amount", F.round(F.rand(seed=3) * 300, 2))
      .write.saveAsTable("sales"))
(spark.range(1_000).withColumnRenamed("id", "product_id")
      .withColumn("category", F.concat(F.lit("cat_"), (F.col("product_id") % 20).cast("string")))
      .write.saveAsTable("products"))
(spark.range(200_000).withColumnRenamed("id", "customer_id")
      .withColumn("segment", F.element_at(F.array(F.lit("A"), F.lit("B"), F.lit("C")),
                                          (F.col("customer_id") % 3 + 1).cast("int")))
      .write.saveAsTable("customers"))

## A. Reading a plan: where are the shuffles?
Look for `Exchange` operators. `Exchange hashpartitioning(key, N)` is a shuffle into N partitions. `BroadcastExchange` sends a small table to every executor instead.

In [ ]:
REVENUE_BY_CATEGORY = """
  SELECT p.category, sum(s.amount) AS revenue
  FROM sales s JOIN products p ON s.product_id = p.product_id
  GROUP BY p.category"""
q = spark.sql(REVENUE_BY_CATEGORY)
q.explain(mode="formatted")

## B. Broadcast join vs sort-merge join
`products` is tiny, so Spark broadcasts it automatically (below `spark.sql.autoBroadcastJoinThreshold`, 10 MB by default). Turn broadcasting off to see the cost of a full shuffle join:

![Broadcast hash join sends a copy of the small table to every executor; sort-merge join shuffles both tables by the join key](../images/3.4-broadcast-vs-sort-merge.svg)

**How to read it:** yellow boxes are what crosses the network. With a broadcast only the 1,000-row `products` table travels;
with sort-merge every `sales` row is hashed by `product_id` and sent to the executor that owns that key.
In the next output this shows up as `join_strategy` and a much larger `shuffle_bytes_written` for the SMJ.

In [ ]:
broadcast = perf.run(q, "broadcast hash join")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)
smj = perf.run(spark.sql(REVENUE_BY_CATEGORY), "sort-merge join (broadcast disabled)")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.sql.adaptive.autoBroadcastJoinThreshold")

display(perf.compare(broadcast, smj))
perf.plot([broadcast, smj], "shuffle_bytes_written", "Shuffle bytes written", highlight="broadcast hash join",
          note="only the aggregation shuffles");

Look at the gap between the two bars: the broadcast plan shuffles only the small aggregate, the sort-merge plan also shuffles all 4 M sales rows.

The sort-merge join shuffles **both** sides, 4 M sales rows included. You can also force a broadcast with a hint when the optimizer's size estimate is wrong: `/*+ BROADCAST(c) */` or `F.broadcast(df)`.

In [ ]:
hinted = spark.sql("""
  SELECT /*+ BROADCAST(c) */ c.segment, count(*) AS orders
  FROM sales s JOIN customers c ON s.customer_id = c.customer_id
  GROUP BY c.segment""")
display(perf.compare(perf.run(hinted, "broadcast hint on 200k-row dimension")))

## C. Shuffle partitions and AQE
`spark.sql.shuffle.partitions` (default 200, set to 8 in this lab) decides how many partitions a shuffle produces. Too few gives huge partitions that can spill. Too many gives tiny tasks and scheduling overhead. **Adaptive Query Execution** measures the real shuffle output and **coalesces** small partitions at runtime:

![Without AQE, 400 tiny shuffle partitions each become a task; AQE merges neighbouring partitions into a few right-sized tasks](../images/3.4-aqe-coalesce.svg)

**How to read it:** each thin bar is one shuffle partition. AQE looks at the real sizes after the map stage and stacks neighbours
until they reach the advisory size. Compare the `tasks` column of the two `stage_summary` outputs below: 400 vs a few.

In [ ]:
agg = spark.table("sales").groupBy("customer_id").agg(F.sum("amount").alias("total"))

for aqe in ("false", "true"):
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    spark.conf.set("spark.sql.shuffle.partitions", 400)
    r = perf.run(agg, f"400 shuffle partitions, AQE={aqe}")
    print(r["query"], "->", r["seconds"], "s")
    display(perf.stage_summary(spark, last=1)[["stage", "tasks", "median_task_ms", "max_task_ms"]])
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", 8)

With AQE off, the final stage runs 400 tiny tasks. With AQE on, it runs only a few. On Databricks, AQE is on by default and `spark.sql.shuffle.partitions = auto` sizes the initial shuffle too.

## D. Reduce what you shuffle
Shuffle volume = rows × width. Select only the needed columns and filter **before** the wide operation. The optimizer pushes simple filters and projections down for you. It **can't** do that across UDFs, `cache()` boundaries, or when you `collect` and rebuild.

In [ ]:
wide = (spark.table("sales").withColumn("payload", F.repeat(F.lit("x"), 200))   # a wide, unused column
        .repartition(16, "customer_id").groupBy("customer_id").agg(F.sum("amount")))
narrow = (spark.table("sales").select("customer_id", "amount")
          .repartition(16, "customer_id").groupBy("customer_id").agg(F.sum("amount")))
display(perf.compare(perf.run(wide, "repartition with wide rows"), perf.run(narrow, "repartition with 2 columns")))

## E. Key takeaways
- Find shuffles with `explain()` (`Exchange`) and the UI (*Shuffle Read/Write*).
- Broadcast small dimensions. Check that the size estimate allows it, or use a hint.
- Keep AQE on. Tune `spark.sql.shuffle.partitions` only for unusually large or small jobs.
- Shuffle less data: prune columns, filter first, aggregate before joining when possible.

In [ ]:
DA.cleanup()